# 00 · Colab 环境检查与数学原型

**本 notebook 现在可以运行的内容：** CPU 数学检查、环境记录、结果原子保存、完整性校验和结果同步。

**它不能证明：** 真实 LLaMA 模型增益、全 Transformer FP 等价、GPTQ 迁移、Blackwell kernel 兼容或 G1/G2 达标。小矩阵 PASS 只是一部分 G0 证据。

建议顺序：先在 CPU 跑完本册，再用 GPU 打开 01；不要因为本册 PASS 就直接租卡运行完整实验矩阵。完整升级门槛见 `docs/GATES_AND_PAPER_CRITERIA.md`。

## 路径与环境配置

CPU 默认路径不下载模型、不调用付费服务、不自动升级 PyTorch。**只上传 ipynb 不够，必须一起提供 src/configs/docs。** 首次在 Colab 上传整个 ZIP（或放 Drive），设置 `KIT_ARCHIVE`，将 `EXTRACT_ARCHIVE=True`，即可用下面单元安全解压；Drive 路径先设置 `MOUNT_DRIVE=True`。如果已经手动解压，则直接设置 `PROJECT_ROOT`。缺少基础 Python 依赖时才把 `INSTALL_BASE_DEPENDENCIES=True`。Colab 断线会丢失 `/content`，需要在阶段完成后同步结果；高频权重与激活 I/O 应留在本地磁盘。

访问受限 LLaMA 权重需要你已有的 Hugging Face 访问权。凭据只通过环境变量或 Colab Secrets 提供，不能写入 notebook、配置、提示词、日志或 Git。

In [1]:
from pathlib import Path
import os, sys, json

# Colab: 解压整个项目到 /content/cellquant_research_kit；不要仅上传 ipynb。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or None
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = False
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = False
SEED = 20261115

In [2]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mounting skipped; outputs remain under RUN_ROOT until copied.")

Drive mounting skipped; outputs remain under RUN_ROOT until copied.


In [3]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

Project: /workspace/scratch/506a2af5b817/cellquant_research_kit
Runs: /workspace/scratch/506a2af5b817/cellquant_research_kit/runs


## 1. 保存可复核环境

`SKIPPED_TORCH_NOT_INSTALLED` 或 `CPU_ONLY` 在这个 notebook 合法，但不能被转换成目标 GPU 兼容性 PASS。只记录依赖版本、GPU 与 CUDA 信息，不遍历环境变量或保存 token。

In [4]:
OUT = PATHS["runs"] / "00_math"
OUT.mkdir(parents=True, exist_ok=True)
env = environment_report()
atomic_json(OUT / "environment.json", env)
print(json.dumps(env, ensure_ascii=False, indent=2))

{
  "schema_version": "1.0",
  "created_at_utc": "2026-11-15T09:43:08.818642+00:00",
  "python": "3.12.14",
  "executable": "/opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "machine": "x86_64",
  "packages": {
    "numpy": "2.3.5",
    "scipy": "1.17.0",
    "torch": null,
    "transformers": null,
    "datasets": null,
    "accelerate": null,
    "safetensors": null,
    "nbformat": "5.11.1",
    "nbclient": "0.11.0",
    "fast-hadamard-transform": null,
    "flash-attn": null,
    "triton": null
  },
  "cuda": {
    "available": false,
    "status": "SKIPPED_TORCH_NOT_INSTALLED"
  },
  "compatibility_claim": "CPU tooling only; GPU kernels require target-device tests"
}


## 2. 运行真实 CPU 参考检查

检查正交性、浮点乘积等价、动态量化胞元代数、筛选和完整事件的数值一致性等。这里使用小矩阵和明确的对称 AbsMax RTN，**不是 7B/8B 的量化实验**。

原型的数值根搜索不是区间算术认证；不能把检查结果写成全正交群全局最优证明。

In [5]:
from cellquant.math_reference import run_reference_checks
math_report = run_reference_checks(OUT, seed=SEED)
print(json.dumps(math_report, ensure_ascii=False, indent=2))
if math_report.get("status") != "PASS":
    raise RuntimeError("数学原型检查未通过：先修复，不进行真实模型实验。")

{
  "schema_version": 1,
  "status": "PASS",
  "seed": 20261115,
  "evidence_kind": "EXECUTED_CPU_SYNTHETIC_REFERENCE",
  "llm_experiment": false,
  "source_type": "synthetic_math",
  "scope": "input_side_single_pair",
  "global_R1_model_implemented": false,
  "W_shape": [
    8,
    16
  ],
  "X_shape": [
    16,
    6
  ],
  "pair": [
    5,
    13
  ],
  "quantizer": {
    "bits": 4,
    "codes": [
      -8,
      7
    ],
    "group_size": 16,
    "symmetric": true,
    "clip_ratio": 1.0,
    "rounding": "ties_to_even"
  },
  "checks": {
    "orthogonality": true,
    "product_equivalence": true,
    "screening_event_equivalence": true,
    "cell_partition": true,
    "analytic_product": true,
    "analytic_loss": true,
    "analytic_budgets": true,
    "actual_acceptance": true,
    "independent_requantization": true,
    "rounding_and_zero": true
  },
  "metrics": {
    "fp64_orthogonality_fro": 2.220481980834306e-16,
    "fp64_product_relative_l2": 1.483649826614371e-16,
    "sc

## 3. 保存边界清晰的状态与可迁移工件

这份状态不会写成 G0/G1 通过证书。完整 Transformer 等价还需分别检查原始 FP、Norm 融合 FP、旋转 FP；主方法的真实层收益、同预算对照和 GPTQ 迁移尚未测量。

In [6]:
status = {
    "schema_version":"1.0", "status":"MATH_REFERENCE_PASS",
    "evidence_kind":"SYNTHETIC_MATHEMATICAL_CHECK",
    "full_transformer_equivalence":"NOT_RUN", "G1":"INSUFFICIENT_EVIDENCE",
    "paper_ready":False,
}
atomic_json(OUT / "run_status.json", status)
files = [p for p in OUT.rglob("*.json") if p.name not in
         {"artifact_manifest.json", "migration_manifest.json"}]
manifest = build_manifest(OUT, files, config={"seed":SEED}, run_kind="math_reference")
atomic_json(OUT / "artifact_manifest.json", manifest)
print(verify_manifest(OUT, manifest))
if PATHS["durable"]:
    print(copy_completed_run(OUT, PATHS["durable"] / "00_math"))
else:
    print("本地结果：", OUT, "。Colab结束前请复制到持久目录或下载。")

{'status': 'PASS', 'missing_files': [], 'changed_files': [], 'config_hash_matches': True, 'scope': 'Artifact integrity only; not model accuracy or hardware compatibility'}
本地结果： /workspace/scratch/506a2af5b817/cellquant_research_kit/runs/00_math 。Colab结束前请复制到持久目录或下载。


## 下一步

1. 如果数学检查 FAIL，停止并把完整 JSON 和日志交给 Codex。
2. 如果数学检查 PASS，进入 01 的真实层诊断。
3. 在 Codex 完成全局 R1 求解器、输出侧处理及 matched-cost 对照之前，01 仍不能产生完整 G1 证据。
4. 不修改门槛来让失败结果通过；需要改研究问题时新建协议版本并记录原因。